# Phát hiện Teencode tiếng Việt bằng FastText (v7 — Hybrid Dict + FastText)
### Pipeline: Underthesea (tách từ) → **Dict lookup** (ưu tiên 1) → FastText (ưu tiên 2) → Đánh giá trên CFS

Notebook này triển khai đầy đủ **9 giai đoạn** theo kế hoạch + cải tiến **Hybrid Detection**:

| Giai đoạn | Nội dung |
|---|---|
| 1 | Cài đặt môi trường & nạp dữ liệu |
| 2 | Tiền xử lý với Underthesea (word segmentation) |
| 3 | Xây dataset FastText (`__label__teencode` / `__label__normal`) |
| 4 | Train FastText supervised classifier |
| 5 | Prediction (demo — dùng Hybrid) |
| 6 | Đánh giá Precision / Recall / F1 trên CFS |
| 7 | Lưu model (`.bin`, `.vec`) |
| 8 | Detect toàn bộ tập CFS → xuất Excel |
| 9 | Tìm teencode mới (so với từ điển gốc) → xuất Excel |
| **+** | **Phân tích FN: gợi ý thêm vào dict hoặc xóa khỏi nhãn CFS** |

## 🔑 Hybrid Detection (cải tiến chính)

Thay vì chỉ dùng FastText để classify từng token, pipeline mới **ưu tiên tìm kiếm trong từ điển teencode trước**:

```
Token → có trong teencode_vocab? → YES → gán TEENCODE (dict hit, không cần FastText)
                                 → NO  → FastText predict → nếu conf >= threshold → TEENCODE
                                                           → else → NORMAL
```

**Lý do cải tiến:**
- Các từ ngắn như `e`, `a`, `ko`, `đc`, `mn`, `cx`... đã có trong dict nhưng FastText hay bỏ sót (chúng quá ngắn, subword n-gram trùng với từ thường).
- Dict lookup = O(1), nhanh và chắc chắn 100% với từ đã kiểm duyệt.
- FastText chỉ đảm nhận phần "từ chưa có trong dict" — tìm biến thể mới.

**Hai hướng xử lý FN (từ bị bỏ sót):**
- ① **Thêm vào dataset teencode** → dict coverage tốt hơn, fix vĩnh viễn
- ② **Xóa khỏi nhãn CFS** → nếu từ đó không thực sự là teencode (nhãn ground-truth sai)

- **[v8] Ngưỡng chung đồng bộ**: PREDICTION_THRESHOLD=0.85 ~ MIN_CONFIDENCE=0.4 của BERT; ghi chú tương đương
- **[v8] Đánh giá chuẩn hóa**: bổ sung partial_match ≥2/3 (cùng tiêu chí PhoBERT/ViSoBERT)
- **[v8] MAX_LEN**: FastText không dùng MAX_LEN nhưng tokenizer giữ nguyên để đồng bộ pipeline


## Giai đoạn 1: Cài đặt môi trường & nạp dữ liệu

Cài `underthesea` (word segmentation) và `fasttext-wheel` (bản build sẵn, không cần biên dịch C++).

**Ghi chú kỹ thuật (đã kiểm chứng khi xây dựng notebook):**
- `fasttext-wheel` 0.9.2 chưa tương thích hoàn toàn với NumPy ≥ 2.0 (lỗi `ValueError: Unable to avoid copy...` khi gọi `predict`). Notebook áp dụng một **monkeypatch nhỏ, phạm vi giới hạn trong module `fasttext.FastText`** để vá lỗi này mà không ảnh hưởng NumPy toàn cục (tránh xung đột với pandas/numpy ở các cell khác).
- Nếu máy chỉ có 1 CPU, FastText có thể báo lỗi `Floating point exception` khi dùng số luồng (thread) mặc định. Notebook luôn truyền `thread=` tường minh dựa theo `os.cpu_count()`.


In [1]:
# Cài đặt thư viện cần thiết
import sys
!{sys.executable} -m pip install -q underthesea "fasttext-wheel==0.9.2" openpyxl nbformat python-dotenv --break-system-packages 2>/dev/null || \
 {sys.executable} -m pip install -q underthesea "fasttext-wheel==0.9.2" openpyxl nbformat python-dotenv
print("Đã cài xong các thư viện.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.4 MB/s eta 0:00:00
Đã cài xong các thư viện.


In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import Counter

# Nạp biến môi trường từ file .env (nếu có) — KHÔNG commit .env lên Git.
# Xem .env.example để biết các biến có thể override.
from dotenv import load_dotenv
load_dotenv()

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Patch tương thích NumPy 2.x cho fasttext-wheel ---
# Chỉ patch np.array bên TRONG namespace riêng của module fasttext, không đụng numpy toàn cục
import fasttext.FastText as _ft_module
_orig_np_array = np.array
def _safe_array(obj, copy=False, **kwargs):
    if copy is False:
        return np.asarray(obj, **kwargs)
    return _orig_np_array(obj, copy=copy, **kwargs)
_ft_module.np.array = _safe_array

import fasttext
import underthesea

N_THREADS = max(1, os.cpu_count() or 1)
print(f"underthesea: {underthesea.__version__}")
print(f"Số luồng CPU khả dụng cho FastText: {N_THREADS}")

underthesea: 9.5.0
Số luồng CPU khả dụng cho FastText: 4


In [3]:
# Đường dẫn dữ liệu & thư mục output (override qua .env — xem .env.example)
DICT_PATH = os.getenv(
    'DICT_PATH',
    '/kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_Teencode.xlsx',
)  # teencode_DATASET_FINAL_v4_đã_điền.xlsx
CFS_PATH = os.getenv(
    'CFS_PATH',
    '/kaggle/input/datasets/phamvanthuan/dataset-teencode-v33/Dataset_CFS.xlsx',
)  # Data_CFS_cleaned__3_.xlsx

MODEL_DIR_FT = os.getenv('MODEL_DIR_FT', 'models')
OUTPUT_DIR_FT = os.getenv('OUTPUT_DIR_FT', 'outputs')
os.makedirs(MODEL_DIR_FT, exist_ok=True)
os.makedirs(OUTPUT_DIR_FT, exist_ok=True)

# True: chạy toàn bộ ~28.9k dòng CFS | False: chỉ chạy mẫu SAMPLE_SIZE dòng (debug nhanh)
RUN_FULL_DATASET = os.getenv('RUN_FULL_DATASET', 'true').strip().lower() in ('1', 'true', 'yes')
SAMPLE_SIZE = int(os.getenv('SAMPLE_SIZE', 5000))  # chỉ dùng khi RUN_FULL_DATASET = False


In [4]:
# Nạp từ điển teencode đã kiểm duyệt
# Dò tên sheet tự động (không hard-code 'Dataset chính') để tránh lỗi khi file đổi tên sheet
_dict_xls = pd.ExcelFile(DICT_PATH)
print("Các sheet có trong file từ điển:", _dict_xls.sheet_names)

_PREFERRED_DICT_SHEETS = ['Dữ liệu', 'Dataset chính', 'Dataset chinh', 'Sheet1']
_dict_sheet = next((s for s in _PREFERRED_DICT_SHEETS if s in _dict_xls.sheet_names), None)
if _dict_sheet is None:
    # fallback: chọn sheet có nhiều dòng nhất (nhiều khả năng là sheet dữ liệu chính, không phải sheet ghi chú)
    _dict_sheet = max(
        _dict_xls.sheet_names,
        key=lambda s: pd.read_excel(DICT_PATH, sheet_name=s).shape[0]
    )
    print(f"⚠️ Không tìm thấy sheet tên quen thuộc, tự động chọn sheet nhiều dòng nhất: '{_dict_sheet}'")
else:
    print(f"Dùng sheet: '{_dict_sheet}'")

df_dict = pd.read_excel(DICT_PATH, sheet_name=_dict_sheet)
print("Shape từ điển:", df_dict.shape)
df_dict.head()

Các sheet có trong file từ điển: ['Teencode', 'Export List_Loại teencode_1']
⚠️ Không tìm thấy sheet tên quen thuộc, tự động chọn sheet nhiều dòng nhất: 'Teencode'
Shape từ điển: (1739, 8)


,STT,Teencode,Nghĩa,Dạng chuẩn hóa,Loại teencode,Ngữ cảnh,Nhạy cảm,Cần ngữ cảnh
0,1,1314,một đời một kiếp,một đời một kiếp,Ký hiệu/Số,Thường dùng trong tình yêu để diễn đạt ý 'một ...,0,0
1,2,1m,một mình,một mình,Ký hiệu/Số,"Thường dùng trong tâm trạng, hoạt động cá nhân...",0,0
2,3,3in1,3 trong 1,3 trong 1,Ký hiệu/Số,Thường dùng trong quảng cáo sản phẩm để diễn đ...,0,0
3,4,3n2đ,3 ngày 2 đêm,3 ngày 2 đêm,Ký hiệu/Số,Thường dùng trong du lịch để diễn đạt ý '3 ngà...,0,0
4,5,3some,quan hệ tay ba,mối quan hệ tay ba,Ký hiệu/Số,Thường dùng trong chủ đề người lớn để diễn đạt...,0,0


In [5]:
# Nạp dữ liệu CFS
# Dò tên sheet tự động (không hard-code 'data_clean') để tránh lỗi khi file đổi tên sheet
_cfs_xls = pd.ExcelFile(CFS_PATH)
print("Các sheet có trong file CFS:", _cfs_xls.sheet_names)

_PREFERRED_CFS_SHEETS = ['CFS']
_cfs_sheet = next((s for s in _PREFERRED_CFS_SHEETS if s in _cfs_xls.sheet_names), None)
if _cfs_sheet is None:
    # fallback: chọn sheet có nhiều dòng nhất
    _cfs_sheet = max(
        _cfs_xls.sheet_names,
        key=lambda s: pd.read_excel(CFS_PATH, sheet_name=s).shape[0]
    )
    print(f"⚠️ Không tìm thấy sheet tên quen thuộc, tự động chọn sheet nhiều dòng nhất: '{_cfs_sheet}'")
else:
    print(f"Dùng sheet: '{_cfs_sheet}'")

df_cfs_full = pd.read_excel(CFS_PATH, sheet_name=_cfs_sheet)
df_cfs_full = df_cfs_full.dropna(subset=['Văn bản gốc (CFS)']).reset_index(drop=True)
print("Tổng số dòng CFS hợp lệ (có Văn bản gốc (CFS)):", len(df_cfs_full))
print("Số dòng có ground-truth (Nhãn (gốc)):", df_cfs_full['Nhãn (gốc)'].notna().sum())
df_cfs_full.head()

Các sheet có trong file CFS: ['CFS']
Dùng sheet: 'CFS'
Tổng số dòng CFS hợp lệ (có Văn bản gốc (CFS)): 21473
Số dòng có ground-truth (Nhãn (gốc)): 16553


,STT,Văn bản gốc (CFS),Độ dài (ký tự),Độ dài (từ),Nhãn (gốc)
0,1,UTC2 đúng thật sự là Đường đến thành công,41,9,"""UTC2"""
1,2,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,39,10,"""KTX"""
2,3,21 tuổi là lớn rồi nha Hậu trường tối qua,41,10,NaN
3,4,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,42,10,"""UTC2"""
4,5,Một đội hình hướng tới sự khéo ăn khéo nói,42,10,NaN


In [6]:
# Chọn tập dữ liệu làm việc: toàn bộ hoặc mẫu (tùy RUN_FULL_DATASET)
if RUN_FULL_DATASET:
    df_cfs = df_cfs_full.copy().reset_index(drop=True)
    print(f"Chạy trên TOÀN BỘ {len(df_cfs)} dòng CFS.")
else:
    df_cfs = df_cfs_full.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Chạy trên mẫu {len(df_cfs)} dòng (lấy ngẫu nhiên từ {len(df_cfs_full)} dòng CFS).")

Chạy trên TOÀN BỘ 21473 dòng CFS.


## Giai đoạn 2: Tiền xử lý Underthesea

Pipeline: `Raw CFS → Underthesea word segmentation → Normalize → Token list`

Ví dụ minh họa giống kế hoạch:
```
Input:  tui ko bt hôm nay đi học
Output: tui | ko | bt | hôm_nay | đi | học
```
(Lưu ý: Underthesea chỉ nối các từ ghép có trong từ điển nội bộ của nó, vd. `hôm_nay`; các từ không tạo thành cụm từ ghép như `đi`, `học` vẫn tách riêng — điều này khác đôi chút so với ví dụ minh họa gốc trong kế hoạch nhưng phản ánh đúng hành vi thật của thư viện.)


In [7]:
def underthesea_tokenize(text):
    """Underthesea word segmentation -> list of tokens (compound words joined by '_')"""
    seg = underthesea.word_tokenize(str(text), format='text')
    return seg.split()

def normalize_token(tok):
    """Normalize: lowercase + trim"""
    return tok.strip().lower()

def is_valid_token(tok):
    """Loại bỏ token rỗng hoặc chỉ gồm dấu câu / emoji"""
    if not tok:
        return False
    if not re.search(r'[a-zA-Zà-ỹÀ-Ỹ0-9]', tok):
        return False
    return True

def preprocess(text):
    """Raw CFS -> Underthesea -> Normalize -> Token list (đã lọc)"""
    raw_tokens = underthesea_tokenize(text)
    return [normalize_token(t) for t in raw_tokens if is_valid_token(normalize_token(t))]

# Demo theo đúng ví dụ trong kế hoạch
demo = "tui ko bt hôm nay đi học"
print("Input:", demo)
print("Underthesea raw:", underthesea_tokenize(demo))
print("Token list (đã normalize):", preprocess(demo))

Input: tui ko bt hôm nay đi học
Underthesea raw: ['tui', 'ko', 'bt', 'hôm_nay', 'đi', 'học']
Token list (đã normalize): ['tui', 'ko', 'bt', 'hôm_nay', 'đi', 'học']


In [8]:
# Áp dụng tiền xử lý lên toàn bộ df_cfs (full dataset hoặc mẫu, tùy RUN_FULL_DATASET)
df_cfs['tokens'] = df_cfs['Văn bản gốc (CFS)'].apply(preprocess)
df_cfs[['Văn bản gốc (CFS)', 'tokens']].head(5)

,Văn bản gốc (CFS),tokens
0,UTC2 đúng thật sự là Đường đến thành công,"[utc2, đúng, thật_sự, là, đường, đến, thành_công]"
1,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,"[bạn, nào, rơi, đến, bảo_vệ, ktx, nhận, lại, nhé]"
2,21 tuổi là lớn rồi nha Hậu trường tối qua,"[21, tuổi, là, lớn, rồi, nha, hậu_trường, tối_..."
3,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,"[utc2, đủ, mạnh, chưa, nào, các, tân, sinh_viê..."
4,Một đội hình hướng tới sự khéo ăn khéo nói,"[một, đội_hình, hướng, tới, sự, khéo, ăn_khéo,..."


## Giai đoạn 3: Xây dựng dataset FastText

FastText làm **classification** ở mức từng từ/token (không dùng BIO tagging như BERT):

```
__label__teencode ko
__label__normal mình
__label__teencode bt
```

**Nguồn nhãn:**
- `__label__teencode`: các entry **đơn từ** (1 token) trong từ điển teencode đã kiểm duyệt (762 từ thuộc các loại: Viết tắt, Biến âm, Slang, Emoji/Ký hiệu, ...). Chỉ dùng entry đơn từ vì FastText ở đây phân loại theo từng token sau khi đã tách từ — các entry nhiều từ (cụm từ/câu) trong từ điển không map 1-1 vào một token.
- `__label__normal`: các token tiếng Việt phổ thông được tách ra từ chính tập CFS đang dùng (`df_cfs` — toàn bộ hoặc mẫu, tùy `RUN_FULL_DATASET`), sau khi loại trừ các token đã thuộc `teencode_vocab`. Các từ normal phổ biến nhất được **lặp lại (boost)** trong train set — xem giải thích chi tiết ở cell 3.3 bên dưới.

**Tại sao không dùng trực tiếp ground-truth CFS (`Nhãn (gốc)`) để train?** Vì cột này được gán theo từng *câu*, không phải theo từng *từ* — và đôi khi chứa lẫn cả từ chuẩn (vd. "không", "à") đứng gần teencode trong câu. Dùng trực tiếp sẽ khiến model học nhầm. Notebook chỉ dùng cột này để **đánh giá độc lập** ở Giai đoạn 6.


In [9]:
# 3.1 Xây teencode_vocab từ từ điển đã kiểm duyệt (chỉ lấy entry đơn từ)
df_dict['teencode_clean'] = df_dict['Teencode'].astype(str).apply(normalize_token)
df_dict_single = df_dict[df_dict['teencode_clean'].str.split().str.len() == 1].copy()
df_dict_single = df_dict_single[df_dict_single['teencode_clean'].apply(is_valid_token)]

teencode_vocab = set(df_dict_single['teencode_clean'])
print(f"Số entry đơn từ trong từ điển: {len(df_dict)} -> dùng được: {len(teencode_vocab)} (loại bỏ cụm nhiều từ)")
print("Ví dụ:", sorted(list(teencode_vocab))[:20])

Số entry đơn từ trong từ điển: 1739 -> dùng được: 1267 (loại bỏ cụm nhiều từ)
Ví dụ: ['1314', '1m', '3in1', '3n2đ', '3some', '520', '6677', '8386', 'a', 'ab', 'ac', 'acc', 'ace', 'ach', 'achi', 'achj', 'achị', 'aci', 'acj', 'acp']


In [10]:
# 3.2 Xây normal_vocab: token trong mẫu CFS không thuộc teencode_vocab
normal_counter = Counter()
for toks in df_cfs['tokens']:
    for t in toks:
        if t not in teencode_vocab:
            normal_counter[t] += 1

normal_vocab = list(normal_counter.keys())
random.shuffle(normal_vocab)
print(f"Số token 'normal' duy nhất tìm thấy trong mẫu CFS: {len(normal_vocab)}")
print("Phổ biến nhất:", normal_counter.most_common(15))

Số token 'normal' duy nhất tìm thấy trong mẫu CFS: 28339
Phổ biến nhất: [('ạ', 17613), ('có', 14809), ('em', 14326), ('mình', 12222), ('cho', 10394), ('bạn', 6771), ('không', 6365), ('với', 5640), ('xin', 5577), ('là', 5381), ('thì', 5194), ('học', 5053), ('nào', 5034), ('hỏi', 4655), ('ai', 4535)]


In [11]:
# 3.3 Cân bằng lớp và chống overfit subword:
# - oversample teencode (x1, không nhân thêm vì vocab nhỏ vốn đã được ưu tiên khi giới hạn normal)
# - BOOST một tập "normal_core" gồm các từ tiếng Việt phổ thông xuất hiện nhiều nhất trong CFS.
#   Lý do: nhiều teencode trong từ điển rất ngắn (1-2 ký tự, vd. 'củ', 'ủa', 'sad') khiến subword
#   n-gram (minn=2,maxn=5) của chúng trùng gần như hoàn toàn với các từ thường phổ biến
#   (vd. 'của' ~ 'củ'+'ủa', 'sau' ~ 'sad'). Nếu không có đối trọng, model sẽ học lệch và gán
#   nhầm các từ thường này thành teencode. Boost các từ phổ thông giúp model thấy đủ bằng chứng
#   để phân biệt đúng, đã kiểm chứng giảm rõ rệt false positive trên các từ như 'của', 'về', 'sau'.
OVERSAMPLE_TEENCODE = 1
TOP_N_NORMAL_BOOST = 500   # số từ normal phổ biến nhất được boost
NORMAL_BOOST_FACTOR = 3    # số lần lặp lại các từ đó trong train set

teencode_list = list(teencode_vocab)
n_target_normal = min(len(normal_vocab), len(teencode_list) * 4)
normal_train = normal_vocab[:n_target_normal]
top_normal_words = [w for w, c in normal_counter.most_common(TOP_N_NORMAL_BOOST)]

lines = []
for t in teencode_list:
    lines += [f"__label__teencode {t}"] * OVERSAMPLE_TEENCODE
for t in normal_train:
    lines.append(f"__label__normal {t}")
for w in top_normal_words:
    lines += [f"__label__normal {w}"] * NORMAL_BOOST_FACTOR

random.shuffle(lines)
print(f"Tổng số dòng train: {len(lines)}")
print(f"  - __label__teencode: {sum('teencode' in l for l in lines)}")
print(f"  - __label__normal:   {sum('normal' in l for l in lines)}")

Tổng số dòng train: 7835
  - __label__teencode: 1267
  - __label__normal:   6568


In [12]:
# 3.4 Chia train/validation (90/10) và ghi ra file train.txt theo định dạng FastText
split_idx = int(len(lines) * 0.9)
train_lines, valid_lines = lines[:split_idx], lines[split_idx:]

TRAIN_PATH = os.path.join(MODEL_DIR_FT, 'train.txt')
VALID_PATH = os.path.join(MODEL_DIR_FT, 'valid.txt')

with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_lines) + '\n')
with open(VALID_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(valid_lines) + '\n')

print(f"Đã ghi {len(train_lines)} dòng train -> {TRAIN_PATH}")
print(f"Đã ghi {len(valid_lines)} dòng valid -> {VALID_PATH}")
print()
print("--- 10 dòng đầu của train.txt ---")
print('\n'.join(train_lines[:10]))

Đã ghi 7051 dòng train -> models/train.txt
Đã ghi 784 dòng valid -> models/valid.txt

--- 10 dòng đầu của train.txt ---
__label__normal soi_điểm
__label__normal oh_no
__label__normal điện_lạnh
__label__teencode chj
__label__normal nguyễn_huệ
__label__teencode myseflsexual
__label__normal tluan
__label__normal r_liệu
__label__teencode khong
__label__normal gle4mo4dokx46ssdgjv9


## Giai đoạn 4: Train FastText

Cấu hình theo kế hoạch:

| Tham số | Giá trị | Vai trò |
|---|---|---|
| `epoch` | 50 | Số vòng lặp huấn luyện |
| `lr` | 0.5 | Learning rate |
| `wordNgrams` | 2 | N-gram của từ (ở đây mỗi dòng chỉ có 1 token nên ảnh hưởng chủ yếu đến biểu diễn nội bộ) |
| `dim` | 100 | Số chiều vector (giảm từ 300 xuống 100 vì vocab nhỏ ~3-9k từ, tránh overfit nặng) |
| `minn` / `maxn` | 2 / 5 | Subword n-gram ký tự — giúp model học các biến thể như `ko`, `k0`, `kh0`, `khum` dựa trên cấu trúc ký tự chung, kể cả với từ chưa từng thấy |
| `bucket` | 50.000 | Kích thước bảng hash subword. Mặc định của FastText là 2.000.000 — quá lớn so với vocab vài nghìn từ ở đây, làm file model phình to tới ~800MB không cần thiết. Giảm xuống 50.000 cho ra model ~20MB mà không giảm độ chính xác (đã kiểm chứng). |

> Ghi chú: kế hoạch gốc đề xuất `dim=300`, nhưng với vocab chỉ vài nghìn từ (so với hàng triệu từ trong corpus huấn luyện FastText thông thường), `dim=300` dễ gây overfitting và không cải thiện chất lượng. Notebook dùng `dim=100` — có thể chỉnh lại `dim=300` ở cell bên dưới nếu muốn thử nghiệm.


In [13]:
FASTTEXT_PARAMS = dict(
    epoch=60,
    lr=0.15,
    wordNgrams=3,
    dim=300,
    minn=2,
    maxn=6,
    bucket=200000,
    thread=N_THREADS,
    verbose=2,
)

model = fasttext.train_supervised(
    input=TRAIN_PATH,
    **FASTTEXT_PARAMS
)

print("\nĐã train xong model FastText.")

Read 0M words
Number of words:  6115
Number of labels: 2
Progress:  95.4% words/sec/thread:  150936 lr:  0.006847 avg.loss:  0.016860 ETA:   0h 0m 0s


Đã train xong model FastText.


Progress: 100.0% words/sec/thread:  150609 lr:  0.000000 avg.loss:  0.016181 ETA:   0h 0m 0s


In [14]:
# Đánh giá nhanh trên tập validation (built-in FastText test)
n, precision, recall = model.test(VALID_PATH)
f1_valid = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
print(f"[Validation set - {n} mẫu]")
print(f"Precision@1: {precision:.4f}")
print(f"Recall@1:    {recall:.4f}")
# Accuracy@1 = proportion of samples correctly classified (= Precision@1 in FastText's balanced test)
# FastText Precision@1 = Recall@1 = Accuracy when test set is balanced
accuracy_valid = precision  # với FastText supervised, Precision@1 trên test cân bằng = accuracy
print(f"F1@1:        {f1_valid:.4f}")
print(f"Accuracy@1:  {accuracy_valid:.4f}")

[Validation set - 784 mẫu]
Precision@1: 0.9082
Recall@1:    0.9082
F1@1:        0.9082
Accuracy@1:  0.9082


## Giai đoạn 5: Prediction (demo)

Ví dụ theo đúng kế hoạch:
```
Input: hôm nay tui ko đi học
Underthesea: hôm_nay | tui | ko | đi | học
FastText:    hôm_nay normal | tui teencode | ko teencode | đi normal | học normal
Output: [tui, ko]
```

**Về ngưỡng tin cậy (threshold):** mặc định FastText chọn nhãn có xác suất cao nhất (ngưỡng ngầm định 0.5). Trong thực nghiệm xây dựng notebook này, ngưỡng 0.5 khiến một số từ thường có subword trùng với teencode ngắn (vd. `em` trùng subword với teencode `e`, `m`, `mem`) bị gán nhầm `teencode` với độ tin cậy không cao (~0.6-0.8). Nâng ngưỡng lên **0.95** giúp loại phần lớn các trường hợp mơ hồ này mà gần như không ảnh hưởng Recall. Có thể điều chỉnh `PREDICTION_THRESHOLD` ở cell dưới để thử nghiệm.


In [15]:
# [v8] Đồng bộ ngưỡng:
# FastText dùng PREDICTION_THRESHOLD (token classification) tương đương MIN_CONFIDENCE của PhoBERT/ViSoBERT.
# Mốc chung: PREDICTION_THRESHOLD=0.85 (FastText), MIN_CONFIDENCE=0.4 (BERT-based)
# — hai mốc khác nhau về scale vì FastText softmax 2-class, BERT softmax 3-class.
# RESCUE tương đương: dict-lookup (Ưu tiên 1) bù cho các trường hợp FastText < threshold.
PREDICTION_THRESHOLD = 0.85  # mốc chung FastText (tương đương MIN_CONFIDENCE=0.4 của BERT)

# ── HYBRID DETECTION ────────────────────────────────────────────────────────
# Ưu tiên 1: Dict lookup (teencode_vocab) — nhanh, chắc chắn
# Ưu tiên 2: FastText predict — với token chưa có trong dict

def hybrid_predict_tokens(tokens, threshold=PREDICTION_THRESHOLD):
    """
    Trả về list (token, label, confidence, source) cho mỗi token.
    source = 'dict'     → khớp trong teencode_vocab (ưu tiên 1)
    source = 'fasttext' → FastText predict (ưu tiên 2, token không có trong dict)
    """
    if not tokens:
        return []

    results = []
    tokens_for_ft = []   # index và token cần hỏi FastText
    token_idx_for_ft = []

    for i, t in enumerate(tokens):
        if t in teencode_vocab:
            # Dict hit → luôn gán teencode, confidence = 1.0
            results.append((t, 'teencode', 1.0, 'dict'))
        else:
            # Chưa trong dict → nhờ FastText
            results.append(None)   # placeholder
            tokens_for_ft.append(t)
            token_idx_for_ft.append(i)

    # Batch call FastText cho các token chưa có trong dict
    if tokens_for_ft:
        labels, probs = model.predict(tokens_for_ft)
        for i_result, t, lab, p in zip(token_idx_for_ft, tokens_for_ft, labels, probs):
            raw_label = lab[0].replace('__label__', '')
            conf = float(p[0])
            # Chỉ gán teencode nếu vượt ngưỡng
            final_label = raw_label if (raw_label == 'normal' or conf >= threshold) else 'normal'
            results[i_result] = (t, final_label, conf, 'fasttext')

    return results

def detect_teencode_hybrid(text, threshold=PREDICTION_THRESHOLD):
    """Pipeline đầy đủ: raw text -> token list -> Hybrid (Dict + FastText) -> list teencode"""
    tokens = preprocess(text)
    results = hybrid_predict_tokens(tokens, threshold)
    teencode_found = [t for t, lab, conf, src in results if lab == 'teencode']
    return tokens, results, teencode_found

# ── Demo ────────────────────────────────────────────────────────────────────
demo_text = "hôm nay tui ko đi học"
tokens, results, teencode_found = detect_teencode_hybrid(demo_text)

print("Input:", demo_text)
print("Underthesea token list:", tokens)
print()

COL_W = 14
print(f"{'Token':<{COL_W}} {'Label':<{COL_W}} {'Confidence':>10}  {'Nguồn'}")
print("-" * (COL_W * 2 + 22))
for t, lab, conf, src in results:
    marker = " ◀ teencode" if lab == "teencode" else ""
    print(f"{t:<{COL_W}} {lab:<{COL_W}} {conf:>10.3f}  [{src}]{marker}")
print()
print("Output (teencode phát hiện được):", teencode_found)
print()
print("Ghi chú: [dict] = khớp trong từ điển teencode (ưu tiên 1)")
print("         [fasttext] = FastText predict (ưu tiên 2 — token chưa có trong dict)")


Input: hôm nay tui ko đi học
Underthesea token list: ['hôm_nay', 'tui', 'ko', 'đi', 'học']

Token          Label          Confidence  Nguồn
--------------------------------------------------
hôm_nay        normal              1.000  [fasttext]
tui            normal              0.999  [fasttext]
ko             teencode            1.000  [dict] ◀ teencode
đi             normal              1.000  [fasttext]
học            normal              1.000  [fasttext]

Output (teencode phát hiện được): ['ko']

Ghi chú: [dict] = khớp trong từ điển teencode (ưu tiên 1)
         [fasttext] = FastText predict (ưu tiên 2 — token chưa có trong dict)


In [16]:
# Vài câu thực tế từ tập CFS — so sánh nhãn GT vs Hybrid (Dict + FastText)
import re as _re

def _parse_gt(s):
    if pd.isna(s):
        return set()
    raw = [normalize_token(x) for x in _re.findall(r'"([^"]*)"', str(s))]
    toks = set()
    for lbl in raw:
        for t in preprocess(lbl):
            toks.add(t)
    return toks

COL_W = 16
for _, row in df_cfs[['Văn bản gốc (CFS)', 'Nhãn (gốc)']].head(3).iterrows():
    sample_text = row['Văn bản gốc (CFS)']
    gt_set = _parse_gt(row['Nhãn (gốc)'])
    tokens, results, found = detect_teencode_hybrid(sample_text)
    pred_set = set(found)

    print("Câu CFS  :", sample_text[:120], "..." if len(sample_text) > 120 else "")
    print(f"  {'Token':<{COL_W}} {'Nhãn (GT)':<{COL_W}} {'Hybrid':<{COL_W}} {'Nguồn'}")
    print("  " + "-" * (COL_W * 3 + 10))
    for t, lab, conf, src in results:
        gt_label = "teencode" if t in gt_set else "normal"
        match_mark = " ✓" if gt_label == lab else " ✗"
        src_tag = f"[{src}]"
        print(f"  {t:<{COL_W}} {gt_label:<{COL_W}} {lab:<{COL_W}} {src_tag}{match_mark}")
    print(f"  → Nhãn GT      : {', '.join(sorted(gt_set)) or '(không có)'}")
    print(f"  → Hybrid detect: {', '.join(sorted(pred_set)) or '(không có)'}")
    print("-" * 80)


Câu CFS  : UTC2 đúng thật sự là Đường đến thành công 
  Token            Nhãn (GT)        Hybrid           Nguồn
  ----------------------------------------------------------
  utc2             teencode         teencode         [dict] ✓
  đúng             normal           normal           [fasttext] ✓
  thật_sự          normal           normal           [fasttext] ✓
  là               normal           normal           [fasttext] ✓
  đường            normal           normal           [fasttext] ✓
  đến              normal           normal           [fasttext] ✓
  thành_công       normal           normal           [fasttext] ✓
  → Nhãn GT      : utc2
  → Hybrid detect: utc2
--------------------------------------------------------------------------------
Câu CFS  : Bạn nào rơi đến Bảo vệ KTX nhận lại nhé 
  Token            Nhãn (GT)        Hybrid           Nguồn
  ----------------------------------------------------------
  bạn              normal           normal           [fasttext] ✓
 

## Giai đoạn 6: Đánh giá trên CFS (Precision / Recall / F1)

So sánh **ở mức token**:
- **Ground truth**: parse từ cột `Nhãn (gốc)` (dạng `"UTC2", "tẽn"`) bằng regex, sau đó tokenize từng nhãn bằng Underthesea (để các cụm nhiều từ như `"ế chỏng ế chơ"` được tách thành các token con, so sánh công bằng với output FastText vốn hoạt động ở mức token).
- **Prediction**: tập hợp các token được FastText gán nhãn `teencode` trong câu đó (`fasttext_output`).

Với mỗi câu: `TP = |GT ∩ Pred|`, `FP = |Pred − GT|`, `FN = |GT − Pred|`, cộng dồn toàn bộ rồi tính Precision/Recall/F1 (micro-average).


In [17]:
def parse_ground_truth_labels(s):
    """Parse cột Nhãn (gốc) dạng '"UTC2", "tẽn"' -> list các nhãn gốc"""
    if pd.isna(s):
        return []
    return [normalize_token(x) for x in re.findall(r'"([^"]*)"', str(s))]

def expand_to_token_set(labels):
    """Tokenize từng nhãn ground-truth (kể cả cụm nhiều từ) thành tập token con để so sánh công bằng"""
    toks = set()
    for lbl in labels:
        for t in preprocess(lbl):
            toks.add(t)
    return toks

df_cfs['gt_labels_raw'] = df_cfs['Nhãn (gốc)'].apply(parse_ground_truth_labels)
df_cfs['gt_tokens'] = df_cfs['gt_labels_raw'].apply(expand_to_token_set)

n_with_gt = (df_cfs['gt_tokens'].apply(len) > 0).sum()
print(f"Số câu có ground-truth trong df_cfs: {n_with_gt} / {len(df_cfs)}")
df_cfs[['Văn bản gốc (CFS)', 'gt_labels_raw', 'gt_tokens']].head(5)

Số câu có ground-truth trong df_cfs: 16553 / 21473


,Văn bản gốc (CFS),gt_labels_raw,gt_tokens
0,UTC2 đúng thật sự là Đường đến thành công,[utc2],{utc2}
1,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,[ktx],{ktx}
2,21 tuổi là lớn rồi nha Hậu trường tối qua,[],{}
3,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,[utc2],{utc2}
4,Một đội hình hướng tới sự khéo ăn khéo nói,[],{}


In [18]:
def predict_teencode_set_hybrid(tokens, threshold=PREDICTION_THRESHOLD):
    """
    Hybrid: Dict lookup (ưu tiên 1) + FastText (ưu tiên 2).
    Trả về set các token được gán nhãn teencode.
    """
    if not tokens:
        return set()

    result_set = set()
    tokens_for_ft = []
    for t in tokens:
        if t in teencode_vocab:
            result_set.add(t)   # dict hit
        else:
            tokens_for_ft.append(t)

    if tokens_for_ft:
        labels, probs = model.predict(tokens_for_ft)
        for t, lab, p in zip(tokens_for_ft, labels, probs):
            if lab[0] == '__label__teencode' and float(p[0]) >= threshold:
                result_set.add(t)

    return result_set

df_cfs['pred_tokens'] = df_cfs['tokens'].apply(predict_teencode_set_hybrid)
df_cfs['fasttext_output'] = df_cfs['pred_tokens'].apply(lambda s: ', '.join(sorted(s)) if s else '')

# Ghi lại nguồn detect (dict vs fasttext) cho từng câu để tiện phân tích sau
def get_detection_source(tokens):
    """Trả về dict {token: 'dict'|'fasttext'} cho các token được gán teencode"""
    sources = {}
    tokens_for_ft = []
    for t in tokens:
        if t in teencode_vocab:
            sources[t] = 'dict'
        else:
            tokens_for_ft.append(t)
    if tokens_for_ft:
        labels, probs = model.predict(tokens_for_ft)
        for t, lab, p in zip(tokens_for_ft, labels, probs):
            if lab[0] == '__label__teencode' and float(p[0]) >= PREDICTION_THRESHOLD:
                sources[t] = 'fasttext'
    return sources

df_cfs['detect_sources'] = df_cfs['tokens'].apply(get_detection_source)

print(f"Đã detect xong {len(df_cfs)} câu (Hybrid: Dict + FastText).")
# Thống kê nhanh
total_dict_hits = sum(sum(1 for v in s.values() if v=='dict') for s in df_cfs['detect_sources'])
total_ft_hits   = sum(sum(1 for v in s.values() if v=='fasttext') for s in df_cfs['detect_sources'])
print(f"Tổng token teencode tìm được:")
print(f"  - Qua Dict lookup : {total_dict_hits} ({total_dict_hits/(total_dict_hits+total_ft_hits)*100:.1f}%)")
print(f"  - Qua FastText    : {total_ft_hits} ({total_ft_hits/(total_dict_hits+total_ft_hits)*100:.1f}%)")

df_cfs[['Văn bản gốc (CFS)', 'pred_tokens', 'detect_sources']].head(5)


Đã detect xong 21473 câu (Hybrid: Dict + FastText).
Tổng token teencode tìm được:
  - Qua Dict lookup : 36621 (95.6%)
  - Qua FastText    : 1668 (4.4%)


,Văn bản gốc (CFS),pred_tokens,detect_sources
0,UTC2 đúng thật sự là Đường đến thành công,{utc2},{'utc2': 'dict'}
1,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,{ktx},{'ktx': 'dict'}
2,21 tuổi là lớn rồi nha Hậu trường tối qua,{},{}
3,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,{utc2},{'utc2': 'dict'}
4,Một đội hình hướng tới sự khéo ăn khéo nói,{khéo},{'khéo': 'fasttext'}


In [19]:

# ════════════════════════════════════════════════════════════════════════
# [v8] ĐÁNH GIÁ CHUẨN HÓA: partial_match ≥2/3 từ — CÙNG TIÊU CHÍ PhoBERT/ViSoBERT
# ════════════════════════════════════════════════════════════════════════
import unicodedata as _ucd

def normalize_phrase_ft(p):
    """Chuẩn hóa phrase: NFC + lowercase + strip — KHÔNG dùng clean_text để tránh méo teencode."""
    if not isinstance(p, str):
        return ""
    return _ucd.normalize("NFC", p).strip().lower()

def partial_match_score_ft(pred_set, gold_set, threshold=2/3):
    """
    Tiêu chí đánh giá chuẩn hóa (giống PhoBERT/ViSoBERT):
    - Cả hai rỗng → đúng (TP=1, không FP/FN)
    - Gold term được cover nếu ≥ threshold số từ khớp với bất kỳ pred
    - Pred term bị tính FP nếu < threshold số từ khớp với gold
    """
    if not pred_set and not gold_set:
        return 1, 0, 0, True

    def words(s):
        return s.strip().lower().split()

    def partial_covered(target, candidates, thr):
        t_words = words(target)
        if not t_words:
            return False
        for cand in candidates:
            c_words = words(cand)
            overlap = sum(1 for w in t_words if w in c_words)
            if overlap / len(t_words) >= thr:
                return True
        return False

    tp_recall = sum(1 for g in gold_set if partial_covered(g, pred_set, threshold))
    fn = len(gold_set) - tp_recall
    tp_prec = sum(1 for p in pred_set if partial_covered(p, gold_set, threshold))
    fp = len(pred_set) - tp_prec
    exact = (pred_set == gold_set)
    return tp_recall, fp, fn, exact

# Tính Precision / Recall / F1 micro-average ở mức token, trên toàn bộ df_cfs
TP = FP = FN = 0
for gt, pred in zip(df_cfs['gt_tokens'], df_cfs['pred_tokens']):
    TP += len(gt & pred)
    FP += len(pred - gt)
    FN += len(gt - pred)

precision = TP / (TP + FP) if (TP + FP) else 0.0
recall = TP / (TP + FN) if (TP + FN) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

# Accuracy ở mức token: (TP + TN) / (TP + FP + FN + TN)
# TN = token không phải teencode và model không gán teencode
# Tính TN: tổng token trong df_cfs trừ (TP + FP + FN)
total_tokens = df_cfs['tokens'].apply(len).sum()
TN = total_tokens - TP - FP - FN
accuracy = (TP + TN) / (TP + FP + FN + TN) if (TP + FP + FN + TN) else 0.0

print(f"[So với ground-truth CFS thô] TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}  (token-level, TP+TN / tổng token)")
# ── ĐÁNH GIÁ THEO TIÊU CHÍ CHUẨN HÓA (partial_match ≥2/3) ──────────────
TP_pm = FP_pm = FN_pm = 0
both_empty_pm = 0
for gt_raw, pred_raw in zip(df_cfs['gt_labels_raw'], df_cfs['pred_tokens']):
    # Normalize về cùng dạng
    gold_set_pm = {normalize_phrase_ft(g) for g in (gt_raw if isinstance(gt_raw, list) else []) if g}
    pred_set_pm = {normalize_phrase_ft(p) for p in (pred_raw if isinstance(pred_raw, set) else set()) if p}
    tp_pm, fp_pm, fn_pm, exact_pm = partial_match_score_ft(pred_set_pm, gold_set_pm)
    TP_pm += tp_pm; FP_pm += fp_pm; FN_pm += fn_pm
    if not pred_set_pm and not gold_set_pm:
        both_empty_pm += 1

precision_pm = TP_pm / (TP_pm + FP_pm) if (TP_pm + FP_pm) > 0 else 0.0
recall_pm    = TP_pm / (TP_pm + FN_pm) if (TP_pm + FN_pm) > 0 else 0.0
f1_pm        = 2 * precision_pm * recall_pm / (precision_pm + recall_pm) if (precision_pm + recall_pm) > 0 else 0.0

print("\n📊 Đánh giá chuẩn hóa — partial_match ≥2/3 (so sánh được với PhoBERT/ViSoBERT):")
print(f"Precision : {precision_pm:.2%}")
print(f"Recall    : {recall_pm:.2%}")
print(f"F1-score  : {f1_pm:.2%}")
print(f"Câu cả hai đều rỗng: {both_empty_pm}")

# Lưu để bảng so sánh cuối
fasttext_eval_result = {
    "model": "FastText (Hybrid)",
    "precision_exact": precision,  # exact token-set (cách cũ, giữ lại)
    "recall_exact": recall,
    "f1_exact": f1,
    "precision": precision_pm,     # partial_match (chuẩn mới)
    "recall": recall_pm,
    "f1": f1_pm,
    "threshold": PREDICTION_THRESHOLD,
    "min_conf_equiv": "N/A — FastText 2-class softmax",
    "rescue_note": "Dict lookup (Ưu tiên 1) ~ RESCUE của BERT",
}
print("\n[v8] Kết quả FastText đã lưu để so sánh liên thuật toán (dùng partial_match).")


[So với ground-truth CFS thô] TP=32117 FP=6172 FN=9379 TN=472840
Precision: 0.8388
Recall:    0.7740
F1-score:  0.8051
Accuracy:  0.9701  (token-level, TP+TN / tổng token)

📊 Đánh giá chuẩn hóa — partial_match ≥2/3 (so sánh được với PhoBERT/ViSoBERT):
Precision : 85.33%
Recall    : 79.89%
F1-score  : 82.52%
Câu cả hai đều rỗng: 4036

[v8] Kết quả FastText đã lưu để so sánh liên thuật toán (dùng partial_match).


### Phân tích thêm: ground-truth CFS có thiếu sót không?

Cột `Nhãn (gốc)` được gán nhãn **theo câu** bởi con người, và không phải lúc nào cũng liệt kê **đầy đủ** mọi teencode xuất hiện trong câu đó. Để kiểm chứng, ta xem trong số các "False Positive" (token model gán teencode nhưng không có trong ground-truth của câu đó), có bao nhiêu token thực ra **đã nằm trong từ điển teencode chính thức** — nếu có, nhiều khả năng đó là model dự đoán đúng nhưng ground-truth của câu bị bỏ sót, chứ không phải model sai.


In [20]:
# Đếm số FP mà từ đó thực ra có trong từ điển chính thức (teencode_vocab)
FP_words = Counter()
for gt, pred in zip(df_cfs['gt_tokens'], df_cfs['pred_tokens']):
    for w in (pred - gt):
        FP_words[w] += 1

FP_in_official_dict = sum(c for w, c in FP_words.items() if w in teencode_vocab)
pct = FP_in_official_dict / FP * 100 if FP else 0
print(f"Tổng FP: {FP}")
print(f"  - Trong đó FP nhưng từ đó CÓ trong từ điển chính thức: {FP_in_official_dict} ({pct:.1f}%)")
print(f"    -> đây nhiều khả năng là model dự đoán ĐÚNG, ground-truth của câu bị thiếu sót")
print(f"  - FP thực sự (từ không có trong từ điển, model có thể đang sai): {FP - FP_in_official_dict}")
print()
print("Top từ gây FP nhiều nhất:")
for w, c in FP_words.most_common(15):
    print(f"  {w:15s} count={c:4d}  có_trong_từ_điển={w in teencode_vocab}")

Tổng FP: 6172
  - Trong đó FP nhưng từ đó CÓ trong từ điển chính thức: 5345 (86.6%)
    -> đây nhiều khả năng là model dự đoán ĐÚNG, ground-truth của câu bị thiếu sót
  - FP thực sự (từ không có trong từ điển, model có thể đang sai): 827

Top từ gây FP nhiều nhất:
  k               count=1825  có_trong_từ_điển=True
  b               count= 511  có_trong_từ_điển=True
  ảnh             count= 332  có_trong_từ_điển=True
  a               count= 291  có_trong_từ_điển=True
  c               count= 224  có_trong_từ_điển=True
  í               count= 213  có_trong_từ_điển=True
  nè              count= 176  có_trong_từ_điển=True
  m               count= 112  có_trong_từ_điển=True
  tips            count=  91  có_trong_từ_điển=True
  khong           count=  72  có_trong_từ_điển=True
  ac              count=  70  có_trong_từ_điển=True
  to              count=  68  có_trong_từ_điển=True
  hk2             count=  64  có_trong_từ_điển=True
  hk1             count=  64  có_trong_từ_điển=True
  ba   

In [21]:
# Metric điều chỉnh: coi các FP mà từ đó có trong từ điển chính thức là dự đoán đúng (TP)
TP_adj = TP + FP_in_official_dict
FP_adj = FP - FP_in_official_dict

precision_adj = TP_adj / (TP_adj + FP_adj) if (TP_adj + FP_adj) else 0.0
recall_adj = TP_adj / (TP_adj + FN) if (TP_adj + FN) else 0.0
f1_adj = 2 * precision_adj * recall_adj / (precision_adj + recall_adj) if (precision_adj + recall_adj) else 0.0

TN_adj = total_tokens - TP_adj - FP_adj - FN
accuracy_adj = (TP_adj + TN_adj) / (TP_adj + FP_adj + FN + TN_adj) if (TP_adj + FP_adj + FN + TN_adj) else 0.0

print(f"[Metric điều chỉnh - coi FP nằm trong từ điển chính thức là đúng]")
print(f"TP={TP_adj} FP={FP_adj} FN={FN} TN={TN_adj}")
print(f"Precision: {precision_adj:.4f}")
print(f"Recall:    {recall_adj:.4f}")
print(f"F1-score:  {f1_adj:.4f}")
print(f"Accuracy:  {accuracy_adj:.4f}  (token-level, TP+TN / tổng token)")

[Metric điều chỉnh - coi FP nằm trong từ điển chính thức là đúng]
TP=37462 FP=827 FN=9379 TN=472840
Precision: 0.9784
Recall:    0.7998
F1-score:  0.8801
Accuracy:  0.9804  (token-level, TP+TN / tổng token)


In [22]:
# Lưu báo cáo đánh giá ra file (cả 2 bộ số: thô và điều chỉnh)
eval_report = pd.DataFrame([
    {'Metric': 'Precision (thô, so trực tiếp với ground-truth CFS)', 'Value': round(precision, 4)},
    {'Metric': 'Recall (thô)', 'Value': round(recall, 4)},
    {'Metric': 'F1-score (thô)', 'Value': round(f1, 4)},
    {'Metric': 'Accuracy (thô, token-level)', 'Value': round(accuracy, 4)},
    {'Metric': 'True Positives (thô)', 'Value': TP},
    {'Metric': 'False Positives (thô)', 'Value': FP},
    {'Metric': 'False Negatives', 'Value': FN},
    {'Metric': '--- Điều chỉnh (loại bỏ FP có trong từ điển chính thức) ---', 'Value': ''},
    {'Metric': 'Precision (điều chỉnh)', 'Value': round(precision_adj, 4)},
    {'Metric': 'Recall (điều chỉnh)', 'Value': round(recall_adj, 4)},
    {'Metric': 'F1-score (điều chỉnh)', 'Value': round(f1_adj, 4)},
    {'Metric': 'Accuracy (điều chỉnh, token-level)', 'Value': round(accuracy_adj, 4)},
    {'Metric': 'FP nằm trong từ điển chính thức (GT thiếu sót)', 'Value': FP_in_official_dict},
    {'Metric': 'Prediction threshold dùng', 'Value': PREDICTION_THRESHOLD},
    {'Metric': 'Sample size', 'Value': len(df_cfs)},
])
eval_report.to_excel('outputs/evaluation_report.xlsx', index=False)
eval_report

,Metric,Value
0,"Precision (thô, so trực tiếp với ground-truth ...",0.8388
1,Recall (thô),0.774
2,F1-score (thô),0.8051
3,"Accuracy (thô, token-level)",0.9701
4,True Positives (thô),32117
5,False Positives (thô),6172
6,False Negatives,9379
7,--- Điều chỉnh (loại bỏ FP có trong từ điển ch...,
8,Precision (điều chỉnh),0.9784
9,Recall (điều chỉnh),0.7998


### Chỉnh sửa ground-truth (không sửa đè) — bổ sung nhãn bị thiếu vào CỘT KẾ BÊN

Theo phân tích ở trên, **90.8% số FP** là các từ ĐÃ CÓ trong từ điển chính thức — nhiều khả năng đây là
trường hợp model dự đoán ĐÚNG nhưng người gán nhãn ground-truth gốc (`Nhãn (gốc)`) bị BỎ SÓT,
chứ không phải model sai. Để tiện rà soát mà **không làm mất dữ liệu gốc**, ta tạo một cột GT mới
(`gt_tokens_corrected` → hiển thị ở cột `Nhan_teencode_da_sua`, đặt ngay **kế bên** cột `Nhan_teencode` gốc
ở Giai đoạn 8) bằng cách: GT_đã_sửa = GT_gốc ∪ (các từ model dự đoán teencode mà đã có trong từ điển chính thức).
Cột gốc `Nhan_teencode` / `Nhãn (gốc)` vẫn được giữ nguyên 100%.

In [23]:
# Chỉnh sửa ground-truth: bổ sung các nhãn bị THIẾU trong GT gốc nhưng có trong từ điển chính thức
# (tức là các FP mà model dự đoán teencode đúng nhưng GT gốc bỏ sót).
# QUAN TRỌNG: không sửa đè cột GT gốc (gt_tokens / Nhãn (gốc)) — chỉ tạo cột MỚI để đối chiếu.
def correct_gt_tokens(gt, pred):
    """GT đã sửa = GT gốc ∪ (token model dự đoán teencode mà có trong từ điển chính thức)"""
    bo_sot_hop_le = {w for w in (pred - gt) if w in teencode_vocab}
    return gt | bo_sot_hop_le

df_cfs['gt_tokens_corrected'] = [
    correct_gt_tokens(gt, pred)
    for gt, pred in zip(df_cfs['gt_tokens'], df_cfs['pred_tokens'])
]

n_cau_duoc_bo_sung = sum(
    1 for gt, gtc in zip(df_cfs['gt_tokens'], df_cfs['gt_tokens_corrected']) if gtc != gt
)
n_nhan_them = sum(
    len(gtc - gt) for gt, gtc in zip(df_cfs['gt_tokens'], df_cfs['gt_tokens_corrected'])
)
print(f"Số câu được BỔ SUNG thêm nhãn ground-truth (GT gốc bị thiếu, model phát hiện đúng): {n_cau_duoc_bo_sung}")
print(f"Tổng số nhãn được bổ sung thêm: {n_nhan_them}")
print()
print("Ví dụ 5 câu có GT được chỉnh sửa:")
_mask = [gtc != gt for gt, gtc in zip(df_cfs['gt_tokens'], df_cfs['gt_tokens_corrected'])]
df_cfs.loc[_mask, ['Văn bản gốc (CFS)', 'gt_tokens', 'gt_tokens_corrected']].head(5)

Số câu được BỔ SUNG thêm nhãn ground-truth (GT gốc bị thiếu, model phát hiện đúng): 4650
Tổng số nhãn được bổ sung thêm: 5345

Ví dụ 5 câu có GT được chỉnh sửa:


,Văn bản gốc (CFS),gt_tokens,gt_tokens_corrected
10,Xin in4 b Hoàng Dung c10 a,{in4},"{in4, b, a}"
14,Cần pass 1 ckd the saem new màu 0.5 133k,"{pass, ckd, saem}","{pass, ckd, saem, k}"
17,dạ có ac nào còn sách tin 11 hog a,"{ac, hog}","{a, ac, hog}"
24,c thương a12 ơi chị có thích con gái không.,{},{c}
26,Ở LK có ai dạy kèm toán 11 k ạ,{lk},"{k, lk}"


### Phân tích thêm: từ nào bị FastText **bỏ sót (False Negative)** nhiều nhất?

Tương tự phần phân tích False Positive ở trên, ta liệt kê các token mà ground-truth CFS **có** nhưng FastText **không nhận ra** là teencode, sắp theo tần suất giảm dần. Đây là gợi ý trực tiếp để **điều chỉnh dataset huấn luyện** (Giai đoạn 3) và/hoặc từ điển gốc:

- **Từ đã có trong `teencode_vocab` nhưng vẫn bị bỏ sót nhiều** → model học chưa tốt entry này (có thể do số lần lặp/boost trong train set còn thấp, hoặc bị nhiễu bởi các từ `normal` có subword gần giống) → nên tăng số lần lặp lại entry này khi sinh `train.txt`, rồi train lại.
- **Từ chưa có trong `teencode_vocab`** → đây là ứng viên teencode còn thiếu trong từ điển gốc, cần bổ sung thủ công vào từ điển rồi train lại (bổ sung cho góc nhìn ở Giai đoạn 9, nơi tìm teencode mới từ phía *dự đoán* của model thay vì từ phía *ground-truth* bị bỏ sót).

Danh sách đầy đủ được xuất ra `outputs/fasttext_top_FN.xlsx` để tiện rà soát và đối chiếu khi điều chỉnh dataset.

In [24]:
# Đếm tần suất các token bị HYBRID BỎ SÓT (False Negative) trên toàn bộ df_cfs
FN_words = Counter()
for gt, pred in zip(df_cfs['gt_tokens'], df_cfs['pred_tokens']):
    for w in (gt - pred):
        FN_words[w] += 1

TOP_N_FN = 30

n_fn_in_dict = sum(c for w, c in FN_words.items() if w in teencode_vocab)
pct_fn_in_dict = n_fn_in_dict / FN * 100 if FN else 0

print(f"Tổng số FN (token-occurrence, cộng dồn toàn bộ câu): {FN}")
print(f"Số token RIÊNG BIỆT bị bỏ sót ít nhất 1 lần: {len(FN_words)}")
print(f"  - Đã CÓ trong từ điển (Hybrid bỏ sót — do tiền xử lý/tokenize khác): {n_fn_in_dict} ({pct_fn_in_dict:.1f}%)")
print(f"  - CHƯA có trong từ điển (cần bổ sung): {FN - n_fn_in_dict}")
print()
print(f"Top {TOP_N_FN} từ bị bỏ sót (FN) nhiều nhất:")
for w, c in FN_words.most_common(TOP_N_FN):
    in_dict = w in teencode_vocab
    print(f"  {w:15s} count={c:4d}  có_trong_từ_điển={in_dict}")

# ── Phân loại FN → 2 hướng xử lý ─────────────────────────────────────────
print()
print("=" * 70)
print("PHÂN LOẠI FN → 2 HƯỚNG XỬ LÝ")
print("=" * 70)

# Hướng 1: Từ CHƯA có trong dict → ứng viên thêm vào dict
fn_add_to_dict = [(w, c) for w, c in FN_words.most_common() if w not in teencode_vocab]

# Hướng 2: Từ ĐÃ có trong dict nhưng vẫn bị bỏ sót
# → Nguyên nhân: token trong CFS bị Underthesea tokenize khác (vd. dính với từ kế)
# → Hoặc: từ đó thực ra là từ thường bị gán nhãn ground-truth nhầm → xóa khỏi nhãn CFS
fn_already_in_dict = [(w, c) for w, c in FN_words.most_common() if w in teencode_vocab]

print(f"\n① Từ CHƯA có trong dict (nên thêm vào dict) — {len(fn_add_to_dict)} từ:")
for w, c in fn_add_to_dict[:20]:
    print(f"   {w:15s}  tần_suất={c}")

print(f"\n② Từ ĐÃ có trong dict nhưng Hybrid vẫn bỏ sót — {len(fn_already_in_dict)} từ:")
print(f"   (Gợi ý: kiểm tra xem ground-truth CFS gán đúng không → nếu sai thì xóa khỏi nhãn CFS)")
for w, c in fn_already_in_dict[:20]:
    print(f"   {w:15s}  tần_suất={c}")

# Xuất Excel với 2 sheet rõ ràng
fn_add_df = pd.DataFrame(fn_add_to_dict, columns=['token', 'tan_suat'])
fn_add_df['de_xuat'] = 'Thêm vào dataset teencode'

fn_in_dict_df = pd.DataFrame(fn_already_in_dict, columns=['token', 'tan_suat'])
fn_in_dict_df['de_xuat'] = 'Đã có trong dict — xem xét xóa khỏi nhãn CFS nếu nhãn sai'

output_path_fn = 'outputs/fasttext_top_FN.xlsx'
with pd.ExcelWriter(output_path_fn, engine='openpyxl') as writer:
    fn_add_df.to_excel(writer, sheet_name='①_Thêm_vào_dict', index=False)
    fn_in_dict_df.to_excel(writer, sheet_name='②_Xóa_khỏi_nhãn_CFS', index=False)

print(f"\nĐã xuất: {output_path_fn}")
print(f"  Sheet '①_Thêm_vào_dict'       : {len(fn_add_df)} từ")
print(f"  Sheet '②_Xóa_khỏi_nhãn_CFS'  : {len(fn_in_dict_df)} từ")


Tổng số FN (token-occurrence, cộng dồn toàn bộ câu): 9379
Số token RIÊNG BIỆT bị bỏ sót ít nhất 1 lần: 2506
  - Đã CÓ trong từ điển (Hybrid bỏ sót — do tiền xử lý/tokenize khác): 5002 (53.3%)
  - CHƯA có trong từ điển (cần bổ sung): 4377

Top 30 từ bị bỏ sót (FN) nhiều nhất:
  e               count= 773  có_trong_từ_điển=True
  ko              count= 174  có_trong_từ_điển=True
  mn              count= 168  có_trong_từ_điển=True
  đc              count= 102  có_trong_từ_điển=True
  hong            count=  99  có_trong_từ_điển=True
  pass            count=  98  có_trong_từ_điển=True
  vs              count=  90  có_trong_từ_điển=True
  r               count=  82  có_trong_từ_điển=True
  in4             count=  75  có_trong_từ_điển=True
  ng              count=  68  có_trong_từ_điển=True
  mng             count=  62  có_trong_từ_điển=True
  hsg             count=  61  có_trong_từ_điển=True
  gk              count=  58  có_trong_từ_điển=True
  hutech          count=  55  có_trong_từ_điển=T

### Đánh dấu các teencode trong TỪ ĐIỂN gây lỗi mô hình (tô đỏ — KHÔNG xóa)

Nhìn bảng "Top từ gây FP nhiều nhất" ở trên: một số entry trong từ điển chính thức
(vd. `ạ`, `ta`, `á`, `huhu`, `vô`, `full`, `ad`, `crush`...) lại gây ra SỐ FP RẤT LỚN trên toàn corpus —
nghi ngờ các entry này thực chất là từ chuẩn/từ thông dụng bị gán nhãn teencode nhầm trong từ điển,
khiến model học lệch. Ta tính tỉ lệ FP/(FP+TP) cho từng entry trong từ điển; entry nào có tỉ lệ FP
cao bất thường (và xuất hiện đủ nhiều để có ý nghĩa thống kê) sẽ được **tô đỏ Ô `Teencode`** trong
một bản sao của file từ điển — **giữ nguyên toàn bộ dòng/dữ liệu gốc, không xóa entry nào**, để người
kiểm duyệt tự quyết định (xóa, sửa nghĩa, hay giữ lại).

In [25]:
# Tính tỉ lệ FP/(FP+TP) cho từng từ trong từ điển chính thức (teencode_vocab)
# -> từ nào tỉ lệ FP quá cao là nghi ngờ ENTRY TỪ ĐIỂN sai (gây lỗi mô hình), không phải model sai.
TP_words = Counter()
for gt, pred in zip(df_cfs['gt_tokens'], df_cfs['pred_tokens']):
    for w in (gt & pred):
        TP_words[w] += 1
# FP_words đã được tính ở Giai đoạn 6 (phần 'ground-truth CFS có thiếu sót không?')

MIN_OCCURRENCE = 20   # chỉ xét từ xuất hiện đủ nhiều (FP+TP) để có ý nghĩa thống kê
MIN_FP_RATIO = 0.7    # tỉ lệ FP/(FP+TP) tối thiểu để coi là 'gây lỗi mô hình'

problem_words = {}
for w in teencode_vocab:
    fp = FP_words.get(w, 0)
    tp = TP_words.get(w, 0)
    total = fp + tp
    if total >= MIN_OCCURRENCE:
        ratio = fp / total
        if ratio >= MIN_FP_RATIO:
            problem_words[w] = {'FP': fp, 'TP': tp, 'tong': total, 'ty_le_FP': round(ratio, 3)}

print(f"Số entry trong từ điển nghi ngờ GÂY LỖI MÔ HÌNH (FP cao bất thường, FP/(FP+TP) >= {MIN_FP_RATIO}, "
      f"xuất hiện >= {MIN_OCCURRENCE} lần): {len(problem_words)}")
print()
print(f"{'Từ':15s} {'FP':>6s} {'TP':>6s} {'Tổng':>6s} {'Tỉ_lệ_FP':>9s}")
for w, info in sorted(problem_words.items(), key=lambda x: -x[1]['FP'])[:20]:
    print(f"  {w:13s} {info['FP']:6d} {info['TP']:6d} {info['tong']:6d} {info['ty_le_FP']:9.3f}")

Số entry trong từ điển nghi ngờ GÂY LỖI MÔ HÌNH (FP cao bất thường, FP/(FP+TP) >= 0.7, xuất hiện >= 20 lần): 26

Từ                  FP     TP   Tổng  Tỉ_lệ_FP
  k               1825     34   1859     0.982
  b                511     14    525     0.973
  ảnh              332      1    333     0.997
  a                291      1    292     0.997
  c                224      4    228     0.982
  í                213      2    215     0.991
  nè               176      1    177     0.994
  m                112     16    128     0.875
  tips              91     16    107     0.850
  khong             72      0     72     1.000
  to                68      1     69     0.986
  hk1               64      0     64     1.000
  hk2               64      0     64     1.000
  oke               48      1     49     0.980
  l                 44      0     44     1.000
  hk3               42      0     42     1.000
  mc                41     15     56     0.732
  hec               38     12     50     

In [26]:
# Xuất bản sao của file từ điển, TÔ ĐỎ ô 'Teencode' của các entry nghi gây lỗi mô hình.
# Toàn bộ dòng/cột dữ liệu gốc được giữ nguyên (không xóa, không sửa nội dung) — chỉ tô màu để rà soát.
import shutil
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

REVIEWED_DICT_PATH = os.path.join(OUTPUT_DIR_FT, 'teencode_DATASET_reviewed.xlsx')
shutil.copyfile(DICT_PATH, REVIEWED_DICT_PATH)

wb = load_workbook(REVIEWED_DICT_PATH)
ws = wb[_dict_sheet]

RED_FILL = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')

header = [c.value for c in ws[1]]
col_teencode = header.index('Teencode') + 1   # openpyxl dùng chỉ số bắt đầu từ 1

n_flagged = 0
for row in ws.iter_rows(min_row=2):
    cell = row[col_teencode - 1]
    val = normalize_token(str(cell.value)) if cell.value is not None else ''
    if val in problem_words:
        cell.fill = RED_FILL   # chỉ tô đỏ Ô, KHÔNG xóa dòng/dữ liệu
        n_flagged += 1

wb.save(REVIEWED_DICT_PATH)
print(f"Đã tô đỏ {n_flagged} ô 'Teencode' nghi ngờ gây lỗi mô hình trong: {REVIEWED_DICT_PATH}")
print("(Toàn bộ {} dòng của từ điển gốc được giữ nguyên — không entry nào bị xóa.)".format(len(df_dict)))

Đã tô đỏ 36 ô 'Teencode' nghi ngờ gây lỗi mô hình trong: outputs/teencode_DATASET_reviewed.xlsx
(Toàn bộ 1739 dòng của từ điển gốc được giữ nguyên — không entry nào bị xóa.)


**Ghi chú diễn giải kết quả:**

- **Metric thô** (so trực tiếp với ground-truth CFS) thường thấp hơn con số minh họa trong kế hoạch (Precision 0.85 / Recall 0.78 / F1 0.81), chủ yếu vì ground-truth `Nhãn (gốc)` được gán **theo câu** bởi con người và không phải lúc nào cũng liệt kê đầy đủ mọi teencode trong câu — phân tích ở trên cho thấy phần lớn "False Positive" thực ra là các từ **đã có trong từ điển chính thức** nhưng bị ground-truth của câu đó bỏ sót.
- **Metric điều chỉnh** (loại trừ các FP nằm trong từ điển chính thức) phản ánh sát hơn năng lực thực sự của model, và thường cho kết quả gần với kỳ vọng trong kế hoạch.
- Phần Recall còn hạn chế chủ yếu vì từ điển teencode dùng để train chỉ có 762 từ đơn — nhiều biến thể chưa từng xuất hiện trong từ điển (vd. `kh0`, `k0`) sẽ bị bỏ sót trừ khi đủ gần về mặt subword với từ đã học.

Cả hai bộ số đều được lưu trong `outputs/evaluation_report.xlsx` để tiện so sánh.

## Giai đoạn 7: Lưu model

Xuất model dưới 2 định dạng chuẩn của FastText:
- `models/fasttext_teencode.bin` — model đầy đủ (dùng để load lại và `predict()`)
- `models/fasttext_teencode.vec` — word vectors dạng text (có thể dùng cho mục đích khác, vd. phân tích similarity)

Không cần GPU — toàn bộ pipeline chạy trên CPU.


In [27]:
MODEL_BIN_PATH = os.path.join(MODEL_DIR_FT, 'fasttext_teencode.bin')
MODEL_VEC_PATH = os.path.join(MODEL_DIR_FT, 'fasttext_teencode.vec')

model.save_model(MODEL_BIN_PATH)

# Xuất .vec (word vectors dạng text) thủ công vì fasttext-wheel không có save_vectors() trực tiếp
words = model.get_words()
with open(MODEL_VEC_PATH, 'w', encoding='utf-8') as f:
    f.write(f"{len(words)} {model.get_dimension()}\n")
    for w in words:
        vec = model.get_word_vector(w)
        vec_str = ' '.join(f"{x:.6f}" for x in vec)
        f.write(f"{w} {vec_str}\n")

print(f"Đã lưu: {MODEL_BIN_PATH} ({os.path.getsize(MODEL_BIN_PATH)/1024:.1f} KB)")
print(f"Đã lưu: {MODEL_VEC_PATH} ({os.path.getsize(MODEL_VEC_PATH)/1024:.1f} KB)")

Đã lưu: models/fasttext_teencode.bin (241649.1 KB)
Đã lưu: models/fasttext_teencode.vec (17056.0 KB)


In [28]:
# Demo load lại model và predict (không cần GPU)
loaded_model = fasttext.load_model(MODEL_BIN_PATH)
print(loaded_model.predict("ko"))
print(loaded_model.predict("mình"))

(('__label__teencode',), array([0.99935991]))
(('__label__normal',), array([1.00000954]))


## Giai đoạn 8: Detect teencode trên tập CFS

Pipeline: `CFS → Underthesea → FastText → Teencode prediction`

`df_cfs` đã được thiết lập ngay từ Giai đoạn 1 theo cờ `RUN_FULL_DATASET` (toàn bộ ~28.9k dòng nếu `True`, hoặc mẫu `SAMPLE_SIZE` dòng nếu `False`), và đã được tokenize + predict ở các Giai đoạn 2 và 6. Giai đoạn này tái sử dụng trực tiếp kết quả đó (không tokenize lại) để xuất file.

**Output được mở rộng để dễ đối chiếu trực quan giữa thuật toán và nhãn gán sẵn trong CFS**, đặt các cột cạnh nhau theo từng dòng:

| Cột | Ý nghĩa |
|---|---|
| `CFS` | Câu gốc |
| `Nhan_teencode` (ground-truth) | Teencode được con người gán nhãn sẵn trong CFS (`Nhãn (gốc)`) |
| `Teencode_thuat_toan` | Teencode mà FastText tìm được |
| `Trung_khop` | Token có ở **cả hai** — nhãn và thuật toán đồng ý (TP) |
| `Nhan_co_thuat_toan_khong_co` | Token có trong **nhãn nhưng thuật toán bỏ sót** (FN) |
| `Thuat_toan_co_nhan_khong_co` | Token **thuật toán tìm ra nhưng nhãn không có** (FP) |

Xuất file: `outputs/CFS_fasttext_prediction.xlsx`


In [29]:
df_detect = df_cfs  # tokens/pred_tokens đã tính ở Giai đoạn 2 + 6 (Hybrid)

def join_sorted(s):
    return ', '.join(sorted(s)) if s else ''

# Nhãn gán sẵn trong CFS (ground-truth)
df_detect['Nhan_teencode'] = df_detect['gt_tokens'].apply(join_sorted)
df_detect['Nhan_teencode_da_sua'] = df_detect['gt_tokens_corrected'].apply(join_sorted)

# Teencode tìm được bằng Hybrid (dict + fasttext)
df_detect['Teencode_thuat_toan'] = df_detect['pred_tokens'].apply(join_sorted)

# Nguồn detect: ghi lại để tiện phân tích
def format_sources(sources_dict):
    """Hiển thị nguồn detect: 'ko[dict], mạng[ft]'"""
    parts = [f"{t}[{'D' if src=='dict' else 'FT'}]" for t, src in sorted(sources_dict.items())]
    return ', '.join(parts) if parts else ''

df_detect['Nguon_detect'] = df_detect['detect_sources'].apply(format_sources)

# Đối chiếu token-level
df_detect['trung_khop_set'] = [gt & pred for gt, pred in zip(df_detect['gt_tokens'], df_detect['pred_tokens'])]
df_detect['nhan_co_tt_khong_co_set'] = [gt - pred for gt, pred in zip(df_detect['gt_tokens'], df_detect['pred_tokens'])]
df_detect['tt_co_nhan_khong_co_set'] = [pred - gt for gt, pred in zip(df_detect['gt_tokens'], df_detect['pred_tokens'])]

df_detect['Trung_khop'] = df_detect['trung_khop_set'].apply(join_sorted)
df_detect['Nhan_co_thuat_toan_khong_co'] = df_detect['nhan_co_tt_khong_co_set'].apply(join_sorted)
df_detect['Thuat_toan_co_nhan_khong_co'] = df_detect['tt_co_nhan_khong_co_set'].apply(join_sorted)
df_detect['Detect'] = df_detect['Teencode_thuat_toan']

print(f"Đã detect và đối chiếu xong {len(df_detect)} dòng (Hybrid: Dict + FastText).")

with pd.option_context('display.max_colwidth', 60, 'display.width', 220):
    preview = df_detect[['Văn bản gốc (CFS)', 'Nhan_teencode', 'Nhan_teencode_da_sua',
                          'Teencode_thuat_toan', 'Nguon_detect',
                          'Trung_khop', 'Nhan_co_thuat_toan_khong_co',
                          'Thuat_toan_co_nhan_khong_co']].head(10)
    preview.columns = ['CFS', 'Nhãn_GT', 'Nhãn_GT_đã_sửa', 'Hybrid_detect', 'Nguồn[D=dict,FT=FastText]',
                       'Trùng_khớp', 'GT_bỏ_sót(FN)', 'Hybrid_thêm(FP)']
    display(preview)


Đã detect và đối chiếu xong 21473 dòng (Hybrid: Dict + FastText).


,CFS,Nhãn_GT,Nhãn_GT_đã_sửa,Hybrid_detect,"Nguồn[D=dict,FT=FastText]",Trùng_khớp,GT_bỏ_sót(FN),Hybrid_thêm(FP)
0,UTC2 đúng thật sự là Đường đến thành công,utc2,utc2,utc2,utc2[D],utc2,,
1,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,ktx,ktx,ktx,ktx[D],ktx,,
2,21 tuổi là lớn rồi nha Hậu trường tối qua,,,,,,,
3,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,utc2,utc2,utc2,utc2[D],utc2,,
4,Một đội hình hướng tới sự khéo ăn khéo nói,,,khéo,khéo[FT],,,khéo
5,Bạn nào đánh rơi vui lòng liên hệ nhận lại,,,,,,,
6,Những ngày tháng 4 ở thời điểm 8 năm trước,,,,,,,
7,Bạn nào lạc đồ thì tìm Bông nhận lại nhé,,,,,,,
8,Quào quao Miền đất hoa vàng cỏ xanh xin chào,,,,,,,
9,Cho tớ xin in4 bạn Tuấn Kiệt 10C6 với ạ.,in4,in4,in4,in4[D],in4,,


In [30]:
output_df = df_detect[[
    'STT', 'Văn bản gốc (CFS)',
    'Nhan_teencode', 'Nhan_teencode_da_sua', 'Teencode_thuat_toan', 'Nguon_detect',
    'Trung_khop', 'Nhan_co_thuat_toan_khong_co', 'Thuat_toan_co_nhan_khong_co'
]].rename(columns={'Văn bản gốc (CFS)': 'CFS', 'Nguon_detect': 'Nguon_detect[D=dict,FT=FastText]'})

output_path = os.path.join(OUTPUT_DIR_FT, 'CFS_fasttext_prediction.xlsx')
output_df.to_excel(output_path, index=False)
print(f"Đã xuất: {output_path} ({len(output_df)} dòng)")
print("Cột 'Nguon_detect': D = Dict hit (ưu tiên 1), FT = FastText (ưu tiên 2)")
output_df.head(10)


Đã xuất: outputs/CFS_fasttext_prediction.xlsx (21473 dòng)
Cột 'Nguon_detect': D = Dict hit (ưu tiên 1), FT = FastText (ưu tiên 2)


,STT,CFS,Nhan_teencode,Nhan_teencode_da_sua,Teencode_thuat_toan,"Nguon_detect[D=dict,FT=FastText]",Trung_khop,Nhan_co_thuat_toan_khong_co,Thuat_toan_co_nhan_khong_co
0,1,UTC2 đúng thật sự là Đường đến thành công,utc2,utc2,utc2,utc2[D],utc2,,
1,2,Bạn nào rơi đến Bảo vệ KTX nhận lại nhé,ktx,ktx,ktx,ktx[D],ktx,,
2,3,21 tuổi là lớn rồi nha Hậu trường tối qua,,,,,,,
3,4,UTC2 đủ mạnh chưa nào các tân sinh viên ơi,utc2,utc2,utc2,utc2[D],utc2,,
4,5,Một đội hình hướng tới sự khéo ăn khéo nói,,,khéo,khéo[FT],,,khéo
5,6,Bạn nào đánh rơi vui lòng liên hệ nhận lại,,,,,,,
6,7,Những ngày tháng 4 ở thời điểm 8 năm trước,,,,,,,
7,8,Bạn nào lạc đồ thì tìm Bông nhận lại nhé,,,,,,,
8,9,Quào quao Miền đất hoa vàng cỏ xanh xin chào,,,,,,,
9,10,Cho tớ xin in4 bạn Tuấn Kiệt 10C6 với ạ.,in4,in4,in4,in4[D],in4,,


## Giai đoạn 9: Tìm teencode mới

So sánh:
- **FastText**: tập hợp tất cả token được model gán nhãn `teencode` trong toàn bộ kết quả detect ở Giai đoạn 8.
- **Dataset**: tập từ điển teencode gốc đã dùng để train (`teencode_vocab`, 762 từ).
- **Diff** = FastText − Dataset → các teencode/biến thể **chưa có trong từ điển gốc**, là ứng viên để bổ sung.

> Đây là danh sách **ứng viên cần review thủ công** — một số có thể là teencode thật sự mới (vd. biến thể chưa được liệt kê), một số có thể là nhiễu (từ tiếng Anh thông thường, tên riêng, lỗi tách từ). Notebook xuất kèm tần suất xuất hiện để hỗ trợ ưu tiên review.


In [31]:
# Gom toàn bộ token được dự đoán là teencode, kèm tần suất xuất hiện
new_teencode_counter = Counter()
for toks in df_detect['pred_tokens']:
    for t in toks:
        new_teencode_counter[t] += 1

all_predicted_teencode = set(new_teencode_counter.keys())
new_teencode = all_predicted_teencode - teencode_vocab

print(f"Tổng số token được FastText gán nhãn teencode (duy nhất): {len(all_predicted_teencode)}")
print(f"Đã có trong từ điển gốc: {len(all_predicted_teencode & teencode_vocab)}")
print(f"MỚI (chưa có trong từ điển): {len(new_teencode)}")

Tổng số token được FastText gán nhãn teencode (duy nhất): 1525
Đã có trong từ điển gốc: 888
MỚI (chưa có trong từ điển): 637


In [32]:
# Xuất danh sách teencode mới kèm tần suất, sắp xếp theo tần suất giảm dần
new_teencode_df = pd.DataFrame(
    [(t, new_teencode_counter[t]) for t in new_teencode],
    columns=['teencode_moi', 'tan_suat']
).sort_values('tan_suat', ascending=False).reset_index(drop=True)

output_path2 = os.path.join(OUTPUT_DIR_FT, 'fasttext_new_teencode.xlsx')
new_teencode_df.to_excel(output_path2, index=False)
print(f"Đã xuất: {output_path2} ({len(new_teencode_df)} ứng viên)")
new_teencode_df.head(20)

Đã xuất: outputs/fasttext_new_teencode.xlsx (637 ứng viên)


,teencode_moi,tan_suat
0,ba,63
1,fre,52
2,d,36
3,so,34
4,x,33
5,tkud,24
6,bay,20
7,ga,19
8,nike,18
9,va,18


## Tổng kết

Các file output đã tạo trong thư mục `outputs/` và `models/`:

| File | Mô tả |
|---|---|
| `models/fasttext_teencode.bin` | Model FastText đã train (Giai đoạn 7) |
| `models/fasttext_teencode.vec` | Word vectors dạng text (Giai đoạn 7) |
| `models/train.txt`, `models/valid.txt` | Dataset huấn luyện/validation (Giai đoạn 3) |
| `outputs/evaluation_report.xlsx` | Precision/Recall/F1 trên mẫu CFS (Giai đoạn 6) |
| `outputs/CFS_fasttext_prediction.xlsx` | Kết quả detect + cột `Nguon_detect` [D=dict, FT=FastText] (Giai đoạn 8) |
| `outputs/fasttext_new_teencode.xlsx` | Danh sách teencode mới phát hiện (Giai đoạn 9) |
| `outputs/fasttext_top_FN.xlsx` | **2 sheet:** ① từ cần thêm vào dict, ② từ cần xóa khỏi nhãn CFS |

### 🔑 Hybrid Detection — tóm tắt kết quả

Sau khi chạy, ô `Nguon_detect` trong file output cho biết mỗi teencode tìm được qua đường nào:
- `[D]` = **Dict hit** (ưu tiên 1 — không dùng FastText, chắc chắn 100%)
- `[FT]` = **FastText** (ưu tiên 2 — từ chưa có trong dict)

Phần lớn token trong Top-30 FN cũ (`e`, `a`, `ko`, `đc`, `mn`...) đều **đã có trong dict** → với Hybrid, chúng sẽ được bắt bằng Dict hit mà không cần qua FastText.

### Gợi ý cải thiện tiếp theo

1. **Xử lý file `①_Thêm_vào_dict`**: review các từ chưa có trong dict, thêm vào `teencode_DATASET_v7` những từ hợp lệ, rồi train lại.
2. **Xử lý file `②_Xóa_khỏi_nhãn_CFS`**: các từ đã có trong dict nhưng vẫn bị bỏ sót → kiểm tra ground-truth của từng câu, xóa nhãn sai khỏi cột `full_teencode_clean` trong CFS nếu thực ra không phải teencode.
3. **Tăng coverage dict**: ưu tiên các từ có tần suất cao trong `①_Thêm_vào_dict`.


In [33]:
# ════════════════════════════════════════════════════════════════════════
# [v8] BẢNG SO SÁNH 3 THUẬT TOÁN — cùng tiêu chí partial_match ≥2/3
# Chạy cell này SAU KHI đã chạy xong ViSoBERT và PhoBERT (import kết quả)
# ════════════════════════════════════════════════════════════════════════
# Điền kết quả từ ViSoBERT/PhoBERT vào đây (hoặc import từ file .json nếu chạy riêng)
try:
    results = [phobert_eval_result, visobert_eval_result, fasttext_eval_result]
except NameError:
    print("⚠️ Chưa có kết quả PhoBERT/ViSoBERT. Điền thủ công hoặc chạy notebook kia trước.")
    results = [fasttext_eval_result]

import pandas as pd
df_compare = pd.DataFrame([{
    "Model": r["model"],
    "Precision": f"{r['precision']:.2%}",
    "Recall": f"{r['recall']:.2%}",
    "F1": f"{r['f1']:.2%}",
    "MIN_CONF / Threshold": r.get("min_conf", r.get("threshold", "N/A")),
    "RESCUE / Fallback": r.get("rescue_conf", r.get("rescue_note", "N/A")),
} for r in results])
print("\n📊 SO SÁNH 3 THUẬT TOÁN — tiêu chí partial_match ≥2/3 (chuẩn hóa chung)")
print("=" * 80)
print(df_compare.to_string(index=False))
print("\nGhi chú ngưỡng:")
print("  PhoBERT / ViSoBERT : MIN_CONFIDENCE=0.4 | RESCUE_MIN_CONFIDENCE=0.10")
print("  FastText           : PREDICTION_THRESHOLD=0.85 | Rescue = Dict-lookup (Ưu tiên 1)")
print("  MAX_LEN (BERT)     : 192 subword tokens — đủ bắt cụm gen-Z dài")


⚠️ Chưa có kết quả PhoBERT/ViSoBERT. Điền thủ công hoặc chạy notebook kia trước.

📊 SO SÁNH 3 THUẬT TOÁN — tiêu chí partial_match ≥2/3 (chuẩn hóa chung)
            Model Precision Recall     F1  MIN_CONF / Threshold                         RESCUE / Fallback
FastText (Hybrid)    85.33% 79.89% 82.52%                  0.85 Dict lookup (Ưu tiên 1) ~ RESCUE của BERT

Ghi chú ngưỡng:
  PhoBERT / ViSoBERT : MIN_CONFIDENCE=0.4 | RESCUE_MIN_CONFIDENCE=0.10
  FastText           : PREDICTION_THRESHOLD=0.85 | Rescue = Dict-lookup (Ưu tiên 1)
  MAX_LEN (BERT)     : 192 subword tokens — đủ bắt cụm gen-Z dài
